<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building RAG Agents with LLMs</b></font></h1>
<h2><b>Notebook 6: </b>임베딩 모델과 시맨틱 추론</h2>
<br>

이전 노트북에서는 대용량 문서 영역으로 나아가, 방대한 관련 콘텐츠를 다루는 데 같은 기법을 적용하는 방법을 배웠습니다! 그 과정에서 특히 즉석 해석 영역에서 기존 기법으로는 여전히 해결할 수 없는 몇 가지 과제를 확인했습니다. 이 노트북에서는 이 목표에 접근하는 데 도움이 되는 다른 접근법인 **임베딩 모델**로 초점을 옮깁니다.


<br>

### **학습 목표:**

- 임베딩(단어, 구, 문서의 수치적 표현)과 이것이 딥러닝 모델이 의미를 처리하는 데 어떻게 기여하는지 익힙니다.

- 이러한 임베딩 모델을 대규모 문서 처리에 적용하여, 기존의 문서 요약 및 지식 추출 방법을 강화하는 법을 배웁니다.

<br>

### **생각해 볼 질문:**

- 특히 일관성 문제나 텍스트 변환 문제를 다룰 때, 임베딩은 문서 청크의 더 깊은 이해에 어떻게 기여하나요?

- 임베딩 모델에서 세부 정보와 계산 효율성 사이의 균형을 어떻게 맞출 수 있을까요? LLM으로 쿼리를 다시 표현하거나 정규화(표준화)하는 방법이 있을까요? 아니면 특수한 작업을 위해 파인튜닝할 수 있을까요?

- 임베딩 모델은 이전에 개발한 running state chain과 지식 베이스를 어떻게 보완할 수 있을까요? *(다음 노트북의 주제)*

<br>

### **환경 설정:**

In [ ]:
## Necessary for Colab, not necessary for course environment
# %pip install -qq langchain langchain-nvidia-ai-endpoints fastembed gradio
# %pip install -qq arxiv pymupdf

# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
pprint = partial(console.print, style=base_style)

----

<br>

## **Part 1:** 임베딩 모델 복습

이 섹션에서는 딥러닝을 이용한 자연어 처리의 개념을 복습하여 임베딩 모델이 무엇인지, 그리고 지금까지 당연하게 사용해 온 도구들과 어떤 관계인지 정의합니다.

<br>

### **잠재 임베딩(Latent Embedding) 이해하기**

잠재 임베딩은 딥러닝 네트워크에서 입력과 출력 사이의 간극을 잇는 중간 지대를 나타냅니다. 예를 들어 [**MNIST 숫자**](https://en.wikipedia.org/wiki/MNIST_database)를 분류하도록 설계된 가벼운 2층 네트워크를 생각해 봅시다. 여기서 입력과 출력은 각각 평탄화된 이미지와 one-hot 확률 벡터일 수 있습니다. 이 구성에서 첫 번째 층이 만들어 내는 값이 이미지의 잠재 임베딩이며, 최적화를 통해 마지막 층이 사용하기 유용한 표현으로 수렴합니다. 이렇게 만들어진 것이 사람이 해석할 수는 없지만 원시 벡터 속성을 활용할 수 있는 **의미적으로 풍부한 임베딩**입니다.

<br>

### **단어 임베딩: 언어 모델의 빌딩 블록**

단어 임베딩은 개별 단어의 고차원 벡터 표현으로, 딥 언어 모델의 근간을 이룹니다. 이 임베딩은 특정 작업에 맞춘 end-to-end 파이프라인 안에서의 최적화 과정을 통해 만들어집니다. 관심 있는 분들을 위한 대표적인 독립 예시로 [**Word2vec**](https://en.wikipedia.org/wiki/Word2vec)이 있습니다. 실제로 언어 모델의 $v$개 토큰 어휘에서 토큰 하나는 토큰 인덱스에서 $d$차원 토큰 임베딩으로 매핑됩니다:

$$\text{Token Index} \in \{0, 1, \cdots, v-1\} \to \text{Token Embedding} \in \mathbb{R}^{d}$$

$n$개 토큰의 시퀀스에 대해서는 이 매핑이 시퀀스 전체로 확장됩니다:

$$\text{Token Sequence} \in \{0, 1, \cdots, v-1\}^{v} \to \text{Embedding Sequence} \in \mathbb{R}^{n\times d}$$

<br>

### **문장/문서 임베딩: 맥락과 의미 포착하기**

문장이나 문서 전체를 다룰 때, 임베딩은 맥락, 의미, 요소 간 상호작용을 포착하는 데 핵심적인 역할을 합니다. 거의 모든 대규모 언어 모델은 이러한 문장/문서 임베딩을 생성하기 위해 **transformer 유사 아키텍처**를 활용합니다. transformer는 최적화 문제에 유용해지는 대로 네트워크가 토큰 단위 정보와 시퀀스 단위 정보를 모두 주고받을 수 있게 해 줍니다.

<br>

### **언어 생성에서의 디코더 모델**

챗봇과 기타 언어 생성 작업에 흔히 사용되는 디코더 모델은 토큰 시퀀스를 입력으로 받는 것으로 시작합니다. 이 토큰들을 잠재 시퀀스로 임베딩하고, 단방향 추론을 적용해 출력 시퀀스의 특정 부분에 집중합니다. 이 집중된, 의미적으로 밀도 높은 지점에서 모델은 시퀀스의 다음 토큰을 예측합니다:

$$$$
$$\text{[ Next-Token Generation ]}$$
$$\text{Embedding Sequence} \in \mathbb{R}^{n\times d} \to \text{Latent Sequence} \in \mathbb{R}^{n\times d}$$
$$(\text{Latent Sequence})[\text{last entry}] \in \mathbb{R}^{d} \to \text{Token Prediction} \in \mathbb{R}^{v}$$
$$$$

이 과정은 토큰 예측을 벡터에서 실제 토큰으로 확정하고, 길이 제한이나 정지 토큰 같은 종료 조건이 충족될 때까지 예측 시퀀스를 쌓아 가며 계속됩니다.

$$$$
$$\text{[ Autoregressive Generation ]}$$
$$(\text{Original + Predicted Embedding Sequence}) \in \mathbb{R}^{(n+1)\times e} \to \text{Token Prediction} \in \mathbb{R}^{v}$$
$$\vdots$$
$$(\text{Original + Predicted Embedding Sequence}) \in \mathbb{R}^{(n+m)\times e} \to \text{Token Prediction} \in \mathbb{R}^{v}$$
$$$$
<br>

### **시퀀스 인코딩을 위한 인코더 모델**

인코더 모델은 양방향 아키텍처를 사용하므로 디코더 모델과는 다른 유형의 작업에 적합합니다. 특히 토큰 또는 시퀀스 예측 같은 작업에 효과적입니다. $c$를 클래스 수 또는 회귀 값의 수라고 하면:

$$$$
$$\text{[ Per-Token Prediction ]}$$
$$\text{Embedding Sequence} \in \mathbb{R}^{n\times d} \to \text{Latent Sequence} \in \mathbb{R}^{n\times d} \to \text{Per-Token Predictions} \in \mathbb{R}^{n\times c}$$

$$$$
$$\text{[ Full-Sequence Prediction ]}$$
$$\text{Embedding Sequence} \in \mathbb{R}^{n\times d} \to \text{Latent Sequence} \in \mathbb{R}^{n\times d}$$
$$(\text{Latent Sequence})[\text{0th entry}] \in \mathbb{R}^{d} \to \text{Sequence Prediction} \in \mathbb{R}^{c}$$

<br>

> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/encoder-decoder.png" width=1200px/>
<!-- > <img src="https://drive.google.com/uc?export=view&id=1lhswkAgb5TlDxezg3qDNZQKbOMGFz7H5" width=1200px/> -->

----

<br>

## **Part 2:** 코스 임베딩 모델 접근하기

이 노트북의 나머지 부분에서는 코스 모델 서버를 통해 LangChain의 `Embeddings` 인터페이스를 사용합니다. 서버는 현재 CPU 모델을 `/v1/embeddings` 뒤에 로드해 두고 있으며, 노트북은 여전히 `embed_query`와 `embed_documents`를 호출하지만 모델 로딩과 배포는 서버에 남아 있습니다.

안정적인 `course/embedding` 식별자는 노트북을 로컬 프로세스나 API 리다이렉트에 묶지 않습니다. 확인하고 싶다면 `scope=all` 카탈로그가 현재 구현을 알려 줍니다. 이 환경에서 MiniLM은 각 입력을 384차원 벡터로 매핑하며 CPU에서 실행됩니다.

Colab에서는 주석 처리된 대안이 FastEmbed로 현재 모델을 구성합니다. 이어지는 임베딩 호출과 검색 실습은 동일하게 유지됩니다.

<br>

### **모델 구성하기**

먼저 노트북 전반에서 사용할 서비스 모델의 이름을 지정한 다음, 그 LangChain 클라이언트를 구성합니다.

In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

embedding_model = "course/embedding"
embedding_model

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

embedder = NVIDIAEmbeddings(
    model=embedding_model,
    base_url="http://llm_client:9000/v1",
)

# In Colab, replace the service client above with:
# from langchain_community.embeddings import FastEmbedEmbeddings
# embedder = FastEmbedEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# ChatNVIDIA.get_available_models()
instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

### **쿼리와 문서를 로컬에서 임베딩하기**

LangChain의 `Embeddings` 인터페이스는 **쿼리**와 **문서**를 임베딩하는 별도의 메서드를 제공합니다. MiniLM에서는 두 메서드 모두 같은 인코더를 사용하며 같은 공간의 벡터를 반환합니다. 메서드 이름은 그 벡터들이 어떻게 사용되는지를 반영합니다. 쿼리는 검색 시점에 임베딩되고, 문서는 보통 벡터 스토어를 준비할 때 배치로 임베딩됩니다.

<br>

#### **쿼리 임베딩**
- **목적**: 간단한 진술이나 질문처럼 짧은 형식 또는 질문 형태의 자료를 임베딩하도록 설계되었습니다.
- **메서드**: 각 쿼리를 개별적으로 임베딩하는 `embed_query`를 사용합니다.
- **검색에서의 역할**: 문서 검색 프레임워크에서 검색(쿼리 과정)을 가능하게 하는 "키" 생성기 역할을 합니다.
- **사용 패턴**: 미리 처리된 문서 임베딩 모음과 비교하기 위해 필요할 때마다 동적으로 임베딩됩니다.

<br>

#### **문서 임베딩**
- **목적**: 문서 청크나 문단처럼 긴 형식 또는 응답 형태의 콘텐츠에 맞춰져 있습니다.
- **메서드**: 문서 배치 처리를 위해 `embed_documents`를 사용합니다.
- **검색에서의 역할**: 검색 시스템의 검색 가능한 콘텐츠를 만드는 "값" 생성기 역할을 합니다.
- **사용 패턴**: 보통 전처리 단계에서 대량으로 임베딩되어, 이후 쿼리를 위한 문서 임베딩 저장소를 만듭니다.

<br>

#### **근본적인 유사성과 실제 적용**

쿼리와 문서는 코사인 유사도로 비교하기 전에 같은 모델로 임베딩되어야 합니다. 들어오는 검색에는 `embed_query`를, 문서 모음을 준비할 때는 `embed_documents`를 사용하세요. 임베딩 모델이 바뀌면 다시 쿼리하기 전에 문서 인덱스를 다시 구축하세요.

<br>

#### **예시 "쿼리"와 "문서"로 탐색하기**

탐색을 시작하고 이 과정들이 실제로 어떻게 동작하는지 이해하기 위해 예시 쿼리와 문서 집합을 살펴봅시다. 이 예시들은 흥미로운 속성을 부각하고 일반 텍스트 추론에 대한 임베딩 모델의 능력을 보여 주기 위해 신중하게 선택되었습니다.

In [ ]:
# Example queries and documents
queries = [
    "What's the weather like in Rocky Mountains?",
    "What kinds of food is Italy known for?",
    "What's my name? I bet you don't remember...",
    "What's the point of life anyways?",
    "The point of life is to have fun :D"
]

documents = [
    "Komchatka's weather is cold, with long, severe winters.",
    "Italy is famous for pasta, pizza, gelato, and espresso.",
    "I can't recall personal names, only provide information.",
    "Life's purpose varies, often seen as personal fulfillment.",
    "Enjoying life's moments is indeed a wonderful approach.",
]

이 문단들을 쿼리 경로 또는 문서 경로로 인코딩할 수 있습니다. 의도된 용도에 따라 메서드 시그니처가 다르므로, 두 옵션의 문법은 약간 다릅니다:

In [ ]:
%%time
# Embedding the queries
q_embeddings = [embedder.embed_query(query) for query in queries]

# Embedding the documents
d_embeddings = embedder.embed_documents(documents)

임베딩을 얻었으니, 검색 작업에서 어떤 문서가 합리적인 답으로 걸렸을지 확인하기 위해 결과에 대해 간단한 유사도 검사를 할 수 있습니다. 항목이 준비되면 아래 코드 블록을 실행해 교차 유사도 행렬을 시각화하세요.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

def plot_cross_similarity_matrix(emb1, emb2):
    # Compute the similarity matrix between embeddings1 and embeddings2
    cross_similarity_matrix = cosine_similarity(np.array(emb1), np.array(emb2))

    # Plotting the cross-similarity matrix
    plt.imshow(cross_similarity_matrix, cmap='Greens', interpolation='nearest')
    plt.colorbar()
    plt.gca().invert_yaxis()
    plt.title("Cross-Similarity Matrix")
    plt.grid(True)

plt.figure(figsize=(8, 6))
plot_cross_similarity_matrix(q_embeddings, d_embeddings)
plt.xlabel("Query Embeddings")
plt.ylabel("Document Embeddings")
plt.show()

# queries = [
#     "What's the weather like in the Rocky Mountains?",
#     "What kinds of food is Italy known for?",
#     "What's my name? I bet you don't remember...",
#     "What's the point of life anyways?",
#     "The point of life is to have fun :D"]
# documents = [
#     "Komchatka's weather is cold, with long, severe winters.",
#     "Italy is famous for pasta, pizza, gelato, and espresso.",
#     "I can't recall personal names, only provide information.",
#     "Life's purpose varies, often seen as personal fulfillment.",
#     "Enjoying life's moments is indeed a wonderful approach."]

----

<br>

## **Part 3: [실습]** 합성이지만 더 현실적인 예제

점수가 높은 셀은 더 그럴듯한 쿼리/문서 쌍에 해당해야 합니다. 다음 플롯에서는 두 메서드를 직접 비교할 수 있도록 문서도 `embed_query`에 통과시킵니다. 일부 임베딩 모델은 쿼리와 문서에 서로 다른 전처리를 적용합니다. MiniLM처럼 대칭 인코더라면 결과가 꽤 비슷하게 유지될 수 있습니다:

In [ ]:
plt.figure(figsize=(8, 6))
plot_cross_similarity_matrix(
    q_embeddings,
    [embedder.embed_query(doc) for doc in documents]
)
plt.xlabel("Query Embeddings (of queries)")
plt.ylabel("Query Embeddings (of documents)")
plt.show()

더 일반적으로, bi-encoder를 사용하면 문서를 한 번만 임베딩하고 저장된 벡터를 각 새 쿼리와 비교할 수 있습니다. 쿼리 경로와 문서 경로가 구분된 모델은 각 경로를 서로 다른 입력 형태에 맞게 최적화할 수도 있습니다. MiniLM은 대칭 인코더를 사용하므로, 이 예제는 별도로 학습된 두 인코더의 차이보다는 검색 워크플로에 초점을 맞춥니다. 입력 자체가 비교에 어떤 영향을 주는지 보기 위해, 문서를 더 긴 형식의 변형으로 확장하여 실험을 다시 해 볼 수 있습니다.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
from pprint import pprint

expound_prompt = ChatPromptTemplate.from_template(
    "Generate part of a longer story that could reasonably answer all"
    " of these questions somewhere in its contents: {questions}\n"
    " Make sure the passage only answers the following concretely: {q1}."
    " Give it some weird formatting, and try not to answer the others."
    " Do not include any commentary like 'Here is your response'"
)

###############################################################################################
## BEGIN TODO

expound_chain = (
    ## TODO: flesh out documents into a more verbose form by implementing the expound_chain 
    ##  which takes advantage of the prompt and llm provided above.
    {}
)

longer_docs = []
for i, q in enumerate(queries):
    ## TODO: Invoke the expound_chain pipeline as appropriate
    longer_doc = ""
    pprint(f"\n\n[Query {i+1}]")
    print(q)
    pprint(f"\n\n[Document {i+1}]")
    print(longer_doc)
    pprint("-"*64)
    longer_docs += [longer_doc]

## END TODO
###############################################################################################

-----

긴 형식의 문서가 마음에 들면, 아래 코드를 실행해 그 임베딩을 원래 쿼리와 비교하세요. MiniLM이 두 메서드에 같은 인코더를 사용하더라도, 다시 쓴 문단이 새로운 단어와 구조를 도입하기 때문에 점수가 바뀔 수 있습니다.

다른 임베딩 모델의 경우, 해당 경로가 다른 전처리나 학습 목표를 적용할 수 있으므로 문서화된 쿼리 및 문서 메서드를 따르세요. 어떤 모델을 선택하든, 저장된 문서 벡터와 이에 비교되는 쿼리 모두에 일관되게 사용하세요.

In [ ]:
## At the time of writing, our embedding model supports up to 2048 tokens...
longer_docs_cut = [doc[:2048] for doc in longer_docs]

q_long_embs = [embedder.embed_query(doc) for doc in longer_docs_cut]
d_long_embs = embedder.embed_documents(longer_docs_cut)

## The difference for any particular example may be very small.
## We've raised the similarity matrix to the power of 5 to try and spot a difference.
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plot_cross_similarity_matrix(q_embeddings, q_long_embs)
plt.xlabel("Query Embeddings (of queries)")
plt.ylabel("Query Embeddings (of long documents)")

plt.subplot(1, 2, 2)
plot_cross_similarity_matrix(q_embeddings, d_long_embs)
plt.xlabel("Query Embeddings (of queries)")
plt.ylabel("Document Embeddings (of long documents)")
plt.show()

<br>

**참고:** 서로 크게 다른 두 bi-encoder 구성 요소를 보고 싶다면 [Contrastive Language-Image Pre-training 모델(CLIP)](https://openai.com/index/clip/)을 살펴보세요. 이 bi-encoder 쌍은 쿼리와 문서 대신 이미지와 텍스트 모달리티를 연결하기 위해 훨씬 넓은 모달리티 간극을 넘어 시너지를 냅니다.

----

<br>

## **Part 4: [실습]** 시맨틱 가드레일을 위한 임베딩

다음 노트북에서는 임베딩 모델을 내부적으로 사용하는 상위 수준의 유틸리티를 사용하기 시작합니다. 그렇지만 원시 메서드가 아직 기억에 생생할 때 탐구할 수 있는 몇 가지 중요한 개념이 있습니다!

특히 프로덕션 모델의 핵심 구성 요소인 **시맨틱 가드레일(semantic guardrailing)** 의 근간으로 사용할 수 있습니다. 구체적으로, 임베딩을 사용해 챗봇이 답하기에 유용하지 않을 가능성이 높은(또는 적극적으로 해로운) 메시지를 걸러낼 수 있습니다!

**이 실습은 [**`64_guardrails.ipynb`**](64_guardrails.ipynb)로 분리되어 있습니다.**

-----

## **Part 5:** 마무리

이 노트북을 마치면 시맨틱 임베딩 모델의 가치 제안에 익숙해지고, 이를 사용해 데이터셋에서 관련 정보를 검색하는 방법을 이해하게 될 것입니다!

### <font color="#76b900">**수고하셨습니다!**</font>

### **다음 단계:**
1. **[선택]** 노트북 상단의 **"생각해 볼 질문" 섹션**을 다시 읽고 가능한 답을 생각해 보세요.
2. **[심화]** 시간이 된다면 시맨틱 가드레일을 다루는 **Notebook 6.4**를 살펴보고 완료해 보세요.
3. **Vectorstore를 이용한 검색**을 다루는 다음 영상으로 이어서 진행하세요.
4. 영상을 본 뒤 **Vectorstore를 이용한 검색**에 해당하는 노트북으로 넘어가세요.

---

<div style="width: 55%%; background-color: white; margin-top: 50px;"><center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png" width="300" /></a></center></div>